In [ ]:
# ============ IMPORTS ============
import pandas as pd
import numpy as np
import optuna
import mlflow
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_validate
import optuna.visualization as vis
from sklearn.metrics import make_scorer, matthews_corrcoef, balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

# ========== CONFIG ==========
sklearn.set_config(enable_metadata_routing=True)
window = 7  # days
cv_n_splits = 5
scoring_metrics = {
    "f1_macro": make_scorer(f1_score, average="macro",   zero_division=0),
    "ba": make_scorer(balanced_accuracy_score),
    "mcc": make_scorer(matthews_corrcoef),
}
feature_set = "confirmed" # "all" or "confirmed" -  features confirmed by boruta algorithm
optuna_n_trials = 100
early_stop_patience = 100
xgboost_device = "cuda"  # change to "cpu" if no GPU available
bounded_features_and_limits = {
    "RSI":      (0,    100),
    "STOCHRSI": (0,    100),
    "STOCH_K":  (0,    100),
    "STOCH_D":  (0,    100),
    "WILLR":    (-100,   0),
    "MFI":      (0,    100),
    "ADX":      (0,    100),
    "ADXR":     (0,    100),
    "PLUS_DI":  (0,    100),
    "MINUS_DI": (0,    100),
    "AROONOSC": (-100, 100),
    "ULTOSC":   (0,    100),
    "CMO":      (-100, 100),
}
# ============================

# ============ LOAD LABELLED DATA ============
labelled_data_cache = pd.read_pickle('data_cache/04_labelled_data.pkl')
X_train = labelled_data_cache['X_train']
y_train = labelled_data_cache['y_train']
profit_target = labelled_data_cache['profit_target']
stop_loss = labelled_data_cache['stop_loss']

# =========== LOAD LATEST FEATURE SELECTION FROM MLFLOW ============
fs_experiment = mlflow.get_experiment_by_name("05_feature_selection")
# List Previous Runs
fs_runs = mlflow.search_runs(experiment_ids=[fs_experiment.experiment_id],
                                    filter_string = "status = 'FINISHED'",
                                    order_by = ['start_time DESC'])
#Load latest run ID
selected_run_id = fs_runs.iloc[0]["run_id"]
# Load artifact
fs_artifact_local_path = mlflow.artifacts.download_artifacts(
    run_id = selected_run_id, 
    artifact_path="05_feature_status.pkl"
)
# Load dataframe
feature_status = pd.read_pickle(fs_artifact_local_path)

# =========== FEATURE SELECTION ============
confirmed_features = feature_status.loc[feature_status['status'] == 'confirmed', 'feature'].tolist()
if feature_set == "confirmed":
    X_train = X_train[confirmed_features]

# ============ LABEL REMAPPING ============
label_map   = {-1: 0, 0: 1, 1: 2}
label_unmap = { 0: -1, 1: 0, 2: 1}
y_train_enc = y_train.map(label_map)

# ============ CV STRATEGY ============
cv_split = TimeSeriesSplit(n_splits=cv_n_splits,gap = window) # 5 sequential train/test windows across the data.

# ============ UNBOUNDED FEATURE SCALING ============
unbounded_feature_cols = [col for col in X_train.columns if col not in bounded_features_and_limits]
        
def robust_scaler():
    return ColumnTransformer(transformers=[
        ("robust", RobustScaler(), unbounded_feature_cols),
    ], remainder="passthrough")  # bounded cols pass through unchanged
    

In [ ]:
# ============ HELPER FUNCTION ============
def evaluate_trial(trial, result):
    train_f1 = result["train_f1_macro"].mean()
    train_mcc = result["train_mcc"].mean()
    train_ba = result["train_ba"].mean()
    test_f1 = result["test_f1_macro"].mean()
    test_mcc = result["test_mcc"].mean()
    test_ba = result["test_ba"].mean()

    trial.set_user_attr("train_f1_macro", round(float(train_f1), 4))
    trial.set_user_attr("train_mcc", round(float(train_mcc),4))
    trial.set_user_attr("train_ba", round(float(train_ba), 4))
    trial.set_user_attr("test_f1_macro", round(float(test_f1), 4))
    trial.set_user_attr("test_mcc", round(float(test_mcc), 4))
    trial.set_user_attr("test_ba", round(float(test_ba), 4))

    return test_mcc

# ============ OBJECTIVE FUNCTIONS ============
def objective_log_reg(trial):
    params = dict(
        C = trial.suggest_float("C", 1e-2, 9e-1, log=True)
    )
    # Low C - heavy regularisation: forces spread across all classes
    # Current suitable C range - 0.01 and 0.9
    model = Pipeline([
        ("preprocessor", robust_scaler()),
        ("clf", LogisticRegression(**params,
                                    solver="saga", max_iter=2000,
                                    class_weight="balanced", random_state=42
        ))
    ]) 
    result   = cross_validate(model, X_train, y_train, cv=cv_split,
                            scoring=scoring_metrics, return_train_score=True)
    return evaluate_trial(trial, result)

def objective_svc(trial):
    params = dict(
        C = trial.suggest_float("C", 1e0, 5e1, log=True),
        gamma = trial.suggest_float("gamma", 3e-3, 1e-1, log=True)
    )
    # Higher C performed better.
    # Current C range is 50, gamma is good as is
    model = Pipeline([
        ("preprocessor", robust_scaler()),
        ("clf", SVC(**params,
                    kernel="rbf",probability=True,
                    class_weight="balanced", random_state=42
        ))
    ])
    
    result   = cross_validate(model, X_train, y_train, cv=cv_split,
                              scoring=scoring_metrics, return_train_score=True)   
    return evaluate_trial(trial, result)

def objective_rf(trial):
    params = dict(
        max_depth = trial.suggest_int("max_depth", 3, 6),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 15, 30),
        max_features = trial.suggest_categorical("max_features",["sqrt","log2"]),
        n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    )
    # Model overfitting, max depth 10 too much with 26 leaves.
    # Larger leaves --> Larger leaves, smaller depth means more generalisation
    model = RandomForestClassifier(**params,
                                        class_weight="balanced",
                                        random_state=42,
                                        n_jobs=-1
        )
    result   = cross_validate(model, X_train, y_train, cv=cv_split,
                              scoring=scoring_metrics, return_train_score=True)
    return evaluate_trial(trial, result)


def objective_xgboost(trial):
    params = dict(
        max_depth = trial.suggest_int("max_depth", 5, 8),
        subsample = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha = trial.suggest_float("reg_alpha", 0.0, 3.0),
        reg_lambda = trial.suggest_float("reg_lambda", 0.5, 3.0),
        min_child_weight = trial.suggest_int("min_child_weight",3,15),
        n_estimators  = trial.suggest_int("n_estimators", 100, 1000, step=50),
    )
    # reg_alpha is L1, reg_lambda is L2
    model = XGBClassifier(**params,
                            learning_rate=0.05,
                            random_state=42,
                            tree_method="hist",
                            device=xgboost_device,
                            n_jobs=1 # not reproducible if using -1
        )
    model.set_fit_request(sample_weight=True)
    sample_weights = compute_sample_weight("balanced", y_train_enc)
    # XGBoost requires 0-indexed labels — use encoded y_train
    result   = cross_validate(model, X_train, y_train_enc, cv=cv_split,
                              scoring=scoring_metrics, return_train_score=True,
                              params={"sample_weight":sample_weights}
                              )
    # The cross_validate result dict is always structured as "test_{key}" and "train_{key}" where {key} is whatever you named it in the scoring dict.
    return evaluate_trial(trial, result)

In [ ]:
# ============ INITIALISE VARIABLES ============
objective_dict = {
    "log_reg": objective_log_reg,
    "svc": objective_svc,
    "random_forest": objective_rf,
    "xgboost": objective_xgboost
}

In [ ]:
# ============ MLFLOW SET UP ============
mlflow.set_experiment("06_hyperparameter_tuning")

# ============ START OPTUNA STUDY ============
for model_name, objective_fn in objective_dict.items():
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    
# ============ EARLY STOPPING CALLBACK ============
    def early_stopping_callback(study, trial):
        best_trial = study.best_trial.number
        if (trial.number - best_trial) >= early_stop_patience:
            study.stop()    

    study.optimize(objective_fn, n_trials=optuna_n_trials, n_jobs=1, show_progress_bar=True, callbacks=[early_stopping_callback])
# ============ MLFLOW LOG ============
    with mlflow.start_run(run_name=f"hyperparameters_{model_name}_{feature_set}_features"):
        # Log X/y_train
        hp_labelled_data = pd.concat([X_train,y_train,y_train_enc],axis=1)
        mlflow.log_input(
            mlflow.data.from_pandas(
                hp_labelled_data,  # single DataFrame with features + labels
                name="hp_labelled_data",
                targets=y_train.name    # tells MLflow which column is the label
            ),
            context="hyperpar tuning"
        )
        hp_labelled_data.to_pickle('data_cache/06_hp_labelled_data.pkl')
        mlflow.log_artifact('data_cache/06_hp_labelled_data.pkl')
        
        # Log parameters
        mlflow.log_param("model", model_name)
        mlflow.log_param("feature_set", feature_set)
        mlflow.log_param("n_trials", len(study.trials))
        mlflow.log_param("objective",  "test_f1")
        mlflow.log_param("scoring", scoring_metrics)
        mlflow.log_params(study.best_params)
        mlflow.log_param("profit_target", profit_target)
        mlflow.log_param("stop_loss", stop_loss)
        
        # Log metrics
        mlflow.log_metric("best_objective", study.best_trial.value)
        mlflow.log_metric("best_test_f1", study.best_trial.user_attrs.get("test_f1_macro",  -1))
        mlflow.log_metric("best_train_f1", study.best_trial.user_attrs.get("train_f1_macro", -1))
        mlflow.log_metric("train_mcc", study.best_trial.user_attrs.get("train_mcc", -1))
        mlflow.log_metric("test_mcc", study.best_trial.user_attrs.get("test_mcc", -1))
        mlflow.log_metric("train_ba", study.best_trial.user_attrs.get("train_ba", -1))
        mlflow.log_metric("test_ba", study.best_trial.user_attrs.get("test_ba", -1))
        mlflow.log_metric("n_failed_trials", sum(1 for t in study.trials if t.value is None))

        # All visualisations as interactive HTML
        plots = {
            "optimisation_history": (vis.plot_optimization_history(study),8,6),
            "optimisation_history_test_ba": (vis.plot_optimization_history(
                study, target=lambda t: t.user_attrs.get("test_ba", 0),
                target_name="test_ba"
            ),8,6),
            "contour": (vis.plot_contour(study),8,6),
            "slice": (vis.plot_slice(study),8,6),
            "param_importances": (vis.plot_param_importances(study),8,6),
            "parallel_coordinate": (vis.plot_parallel_coordinate(study),8,6),
            "rank": (vis.plot_rank(study),8,6),
            "edf": (vis.plot_edf(study),8,6),
            "timeline": (vis.plot_timeline(study),8,6)
        }
        for plot_name, (fig, w, h) in plots.items():
            imgWidth, imgHeight = int((w/2.54)*150), int((h/2.54)*150)
            html_path = f"data_cache/hp_{model_name}_{feature_set}_{plot_name}.html"
            png_path  = f"data_cache/hp_{model_name}_{feature_set}_{plot_name}.png"
            fig.write_html(html_path)
            fig.write_image(png_path, width=imgWidth, height=imgHeight)
            mlflow.log_artifact(html_path)
            mlflow.log_artifact(png_path)
        
        # Store Trials dataframe
        hp_trials_df = study.trials_dataframe().set_index("number")
        hp_trials_df.to_pickle(f"data_cache/06_hp_trials_{model_name}_{feature_set}.pkl")
        mlflow.log_artifact(f"data_cache/06_hp_trials_{model_name}_{feature_set}.pkl")
        
        # Store Best Parameters
        best_params = pd.Series(study.best_params, name=f"{model_name}_{feature_set}")
        best_params.to_pickle(f"data_cache/06_hp_best_params_{model_name}_{feature_set}.pkl")
        mlflow.log_artifact(f"data_cache/06_hp_best_params_{model_name}_{feature_set}.pkl")

    print(f"Best objective: {study.best_trial.value:.4f}")
    print(f"Best params: {study.best_params}")